# (7) Results

This chapter loads the out-of-fold predictions saved by `run_phase5_envelope`
and `run_phase5_map` and regenerates the result figures directly from the
stored prediction arrays — no static PNG dependence, every plot is reproducible
from the saved `.npz` files.

Headline numbers (5-fold CV on **2511 designs**):

| Network | Target | OOF $R^2$ | RMSE | MAPE |
|---|---|---|---|---|
| `EnvelopeMLP` | $T_\text{max}$ | **0.999** | 0.41 Nm | **5.86%** |
| `MapDecoder` (baseline)  | $\eta$ grid (masked) | 0.913 | 0.083 | — |
| `MapDecoder` (depth+1)   | $\eta$ grid (masked) | **0.963** | 0.054 | — |
| `MapDecoder` (width-2×)  | $\eta$ grid (masked) | **0.961** | 0.056 | — |
| `MapDecoder` (width-4×)  | $\eta$ grid (masked) | **0.973** | **0.046** | — |

$N_\text{max}$ is constant 8000 RPM in this dataset (all designs share the same DC-bus
voltage and current limit), so the envelope net's second output is reported but not
predictive in the strict sense. The T-max MAPE dropped from 12.0% (at 759
designs) to 5.86% (at 2511 designs). The baseline η-map decoder's $R^2 = 0.913$ on the
same 2511 designs identified an **architectural** bottleneck rather than a data
bottleneck (§7.7); the architecture sweep (§4.7.3) then lifted the η-map $R^2$ to
**0.973** without changing the training set. The width-4× weights are the new
production decoder for downstream chapters.


## 7.1  Load the OOF predictions


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path('data_gen/outputs')
env  = np.load(ROOT / 'phase5_envelope_train' / 'predictions.npz', allow_pickle=False)
mp   = np.load(ROOT / 'phase5_map_train'      / 'predictions.npz', allow_pickle=False)

print('envelope predictions:')
for k in env.files:
    print(f"  {k:<14s}{env[k].shape}")
print('\nmap predictions:')
for k in mp.files:
    print(f"  {k:<14s}{mp[k].shape}")


## 7.2  Envelope net — predicted vs actual $T_\text{max}$

Each dot is one design held out in its fold. The line is $y = x$ — a perfect
predictor would put every dot on it.


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_percentage_error

t_true = env['oof_true'][:, 1]
t_pred = env['oof_pred'][:, 1]
n_true = env['oof_true'][:, 0]
n_pred = env['oof_pred'][:, 0]

r2_T   = r2_score(t_true, t_pred)
mape_T = mean_absolute_percentage_error(t_true, t_pred) * 100
rmse_T = float(np.sqrt(np.mean((t_pred - t_true)**2)))

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

ax = axes[0]
lim = [0, max(t_true.max(), t_pred.max()) * 1.05]
ax.scatter(t_true, t_pred, s=10, alpha=0.4)
ax.plot(lim, lim, 'k--', lw=1)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('True $T_\\mathrm{max}$ (Nm)')
ax.set_ylabel('Predicted $T_\\mathrm{max}$ (Nm)')
ax.set_title(f'$T_\\mathrm{{max}}$  — $R^2$={r2_T:.3f}, MAPE={mape_T:.1f}%, RMSE={rmse_T:.2f} Nm')
ax.grid(alpha=0.3)

ax = axes[1]
resid = t_pred - t_true
ax.hist(resid, bins=40, edgecolor='black', alpha=0.7)
ax.axvline(0, color='k', lw=1)
ax.set_xlabel('Residual (Nm)  —  $\\hat T_\\mathrm{max} - T_\\mathrm{max}$')
ax.set_ylabel('Designs')
ax.set_title(f'Residual distribution  (median |err| = {np.median(np.abs(resid)):.2f} Nm)')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nN_max: predicted unique values = {np.unique(np.round(n_pred, 1))[:5]}')
print(f'N_max: true unique values      = {np.unique(np.round(n_true, 1))}')


The residuals are tightly centred on zero with no obvious bias. The few outliers
are designs with very low $T_\text{max}$ (under 2 Nm) where the absolute error of
$\sim 0.5$ Nm is small but the percentage is large — the dominant source of the
5.86% MAPE.


## 7.3  Map net — predicted vs ground-truth efficiency maps

Four designs picked across the spread of $T_\text{max}$ values: smallest, mid-low,
mid-high, largest. Each row shows ground truth, prediction, and the absolute
error map (cells outside the operating envelope are masked white).


In [ ]:
y_true = mp['oof_true']
y_pred = mp['oof_pred']
mask   = mp['mask']

master = np.load(ROOT / 'dataset_train' / 'master_dataset.npz', allow_pickle=False)
N_axis = master['N_norm_grid']
T_axis = master['T_norm_grid']
T_max_per_design = master['T_max_Nm']

order = np.argsort(T_max_per_design)
picks = [order[0], order[len(order)//3], order[2*len(order)//3], order[-1]]

fig, axes = plt.subplots(len(picks), 3, figsize=(14, 4*len(picks)))
for row, idx in enumerate(picks):
    yt = np.where(mask[idx], y_true[idx], np.nan)
    yp = np.where(mask[idx], y_pred[idx], np.nan)
    err = np.abs(yp - yt)
    Tm = T_max_per_design[idx]

    for col, (img, title, cmap, vmax) in enumerate([
        (yt,  f'truth  ($T_\\mathrm{{max}}$={Tm:.1f} Nm)', 'viridis', 1.0),
        (yp,  'predicted',                                  'viridis', 1.0),
        (err, 'abs error',                                  'magma',   0.3),
    ]):
        ax = axes[row, col]
        im = ax.imshow(img, origin='lower', aspect='auto',
                       extent=[N_axis[0], N_axis[-1], T_axis[0], T_axis[-1]],
                       cmap=cmap, vmin=0, vmax=vmax)
        ax.set_xlabel('$N / N_\\mathrm{max}$')
        ax.set_ylabel('$T / T_\\mathrm{max}$')
        ax.set_title(title)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


Shape of the predicted maps matches the ground truth: low-speed/high-torque corner
is darker (Cu loss dominates), mid-band plateau is bright (η > 0.9 region), high-speed
drop matches. Absolute error is mostly concentrated at the boundary near the operating
envelope — cells that are sometimes inside and sometimes outside across designs.


## 7.4  Per-design error distribution

How does masked RMSE vary across the 2511 designs?


In [ ]:
# Per-design masked RMSE
per_design_rmse = []
for i in range(y_true.shape[0]):
    m = mask[i]
    if m.sum() == 0:
        per_design_rmse.append(np.nan); continue
    diff = (y_pred[i] - y_true[i])[m]
    per_design_rmse.append(float(np.sqrt(np.mean(diff**2))))
per_design_rmse = np.array(per_design_rmse)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ax = axes[0]
ax.hist(per_design_rmse, bins=40, edgecolor='black', alpha=0.7)
ax.axvline(np.nanmedian(per_design_rmse), color='r', lw=1.5,
           label=f'median = {np.nanmedian(per_design_rmse):.3f}')
ax.set_xlabel('Per-design masked RMSE (η units)')
ax.set_ylabel('Designs')
ax.set_title('Per-design η-map error distribution')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.scatter(T_max_per_design, per_design_rmse, s=8, alpha=0.4)
ax.set_xlabel('$T_\\mathrm{max}$ (Nm)')
ax.set_ylabel('Per-design masked RMSE')
ax.set_title('Error vs design peak torque')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'median per-design RMSE: {np.nanmedian(per_design_rmse):.4f}')
print(f'p90    per-design RMSE: {np.nanpercentile(per_design_rmse, 90):.4f}')
print(f'worst  per-design RMSE: {np.nanmax(per_design_rmse):.4f}')


Error is roughly stationary across the $T_\text{max}$ axis — no systematic bias
against very-low-torque or very-high-torque designs. The 90th-percentile RMSE
stays comfortably under 0.15.


## 7.5  Per-fold metrics (sanity check)


In [ ]:
env_cv = pd.read_csv(ROOT / 'phase5_envelope_train' / 'cv_metrics.csv')
map_cv = pd.read_csv(ROOT / 'phase5_map_train'      / 'cv_metrics.csv')

print('Envelope CV metrics:')
print(env_cv.to_string(index=False))
print('\nMap CV metrics:')
print(map_cv.to_string(index=False))


Per-fold metrics are consistent — no fold catastrophically out of line with the
others, indicating the result is genuine generalisation and not a lucky split.


## 7.6  Training curves

The loss-curve plots are written by the runners alongside the predictions:

```{figure} data_gen/outputs/phase5_envelope_train/loss_curves.png
---
name: fig-loss-envelope
width: 90%
---
Envelope-net training and validation loss across the 5 folds (1500 epochs each).
Curves overlap closely → no overfitting; both still decreasing slightly at 1500 epochs,
but the marginal improvement is too small to matter.
```

```{figure} data_gen/outputs/phase5_map_train/loss_curves.png
---
name: fig-loss-map
width: 90%
---
Map-decoder training and validation loss across the 5 folds (1500 epochs each).
Baseline (157 k params) topology. Train/val gap stays tight throughout — the
1e-4 weight decay is more than needed but cheap insurance. The early plateau in
these curves is the architectural-capacity ceiling that motivates §4.7.2 / §7.7.
```

```{figure} data_gen/outputs/phase5_map_train_arch/width4x/loss_curves.png
---
name: fig-loss-map-width4x
width: 90%
---
Width-4× variant (633 k params, `bottleneck_shape=(32, 15, 20)`,
`decoder_channels=(16, 1)`). All 5 folds converge to substantially lower val loss
(~0.0021 vs ~0.0069 for baseline) and are tightly bunched. This is the
production decoder going forward.
```


## 7.7  How much did more data help, and how much did architecture help?

Two independent scaling experiments tell the full story. First, dataset size at fixed
architecture (baseline 157 k-param `MapDecoder`):

| Metric | 42 designs (smoke) | 759 designs (v1) | 2511 designs (current) | $\Delta$ 759→2511 |
|---|---|---|---|---|
| Envelope $T_\text{max}$ MAPE | 47%   | 12.0% | **5.86%** | $-52\%$ |
| Envelope $T_\text{max}$ RMSE | 4.2 Nm| 0.78 Nm | **0.41 Nm** | $-47\%$ |
| Map masked $R^2$             | 0.69  | 0.912 | **0.913** | $+0.001$ |
| Map masked RMSE (η)          | 0.18  | 0.084 | **0.083** | $-1\%$ |

Then architecture variant at fixed dataset (the 2511-design master, same
hyperparameters, same seed):

| Variant | Parameters | OOF $R^2$ | RMSE (η) | $\Delta R^2$ vs baseline |
|---|---:|---:|---:|---:|
| baseline                | 157 k | 0.913 | 0.083 | — |
| depth+1 (intermediate Conv2d) | 157 k | **0.963** | **0.054** | $+0.050$ |
| width-2×                | 314 k | **0.961** | **0.056** | $+0.048$ |
| width-4×                | 633 k | **0.973** | **0.046** | $+0.060$ |

::::{admonition} Dataset-scaling vs capacity-scaling — confirmed asymmetry
:class: tip

The envelope net (1.3 k params) and the map decoder (157 k params) consumed the same
extra FEMM samples but responded differently:

- **Envelope net — data regime.** 1.3 k parameters are plenty to express
  $(\text{geometry}) \mapsto T_\text{max}$; the bottleneck at 759 designs was
  *sample noise*. Tripling the dataset cut the OOF MAPE by half — a textbook
  data-limited behaviour.
- **Baseline map decoder — capacity regime.** 157 k parameters and two `ConvTranspose`
  upsampling stages are *not* enough to express the full $\eta(N, T)$ shape across
  the design population. Tripling the dataset moved the masked $R^2$ by 0.001 —
  inside run-to-run variance. The architecture-sweep table directly confirms this:
  with the same 2511 designs, swapping the topology lifts $R^2$ from 0.913 to 0.973,
  a $-44\%$ reduction in RMSE that no additional data was needed to unlock.

**The depth+1 result is the elegant finding.** A single same-resolution
`Conv2d(c, c, k=3)` inserted between the two `ConvTranspose` stages — at *identical
parameter count to baseline* — captures essentially the same gain as doubling all
channel widths (width-2×). So the bottleneck was spatial-refinement depth, not
channel breadth. The cells the baseline got most wrong (boundary, low-speed/high-torque
corner; see Chapter 6) line up with the regions a single intermediate conv block can
sharpen.

**Diminishing returns above $R^2 \approx 0.97$.** width-4× costs 2× the parameters of
width-2× for only $+0.012$ $R^2$. Further gains would require qualitative architectural
changes (attention head, U-Net residual decoder), scoped as future work in §4.7.2.
::::

Wall-clock at 2511 designs on CPU: envelope ~40 s, baseline map decoder ~30 min,
width-4× map decoder ~2 h 24 min (the slowest variant in the sweep). Still no GPU
dependency.


In [ ]:
# §7.7 — per-design residual comparison: baseline vs width-4x MapDecoder.
# `mp` is already loaded in cell 2 from phase5_map_train (baseline). We load
# the width-4x predictions here. Per-design RMSE is computed vectorised over
# the design axis to keep this cell fast (Python loops over 2511 designs were
# 50x slower under nbconvert + WSL I/O).
w4x = np.load(ROOT / 'phase5_map_train_arch' / 'width4x' / 'predictions.npz',
              allow_pickle=False)
assert mp['oof_true'].shape == w4x['oof_true'].shape, 'shape mismatch'

def per_design_rmse_vec(pred, true, mask):
    diff = pred - true
    sq = np.where(mask, diff * diff, 0.0)
    cnt = mask.sum(axis=(1, 2)).astype(np.float64)
    sums = sq.sum(axis=(1, 2))
    return np.sqrt(np.divide(sums, cnt,
                             out=np.full_like(cnt, np.nan), where=cnt > 0))

rmse_base = per_design_rmse_vec(mp['oof_pred'],  mp['oof_true'],  mp['mask'])
rmse_w4x  = per_design_rmse_vec(w4x['oof_pred'], w4x['oof_true'], w4x['mask'])

# Pick the worst-on-baseline design as a high-leverage example.
i_worst = int(np.nanargmax(rmse_base))
print(f'Picked design idx {i_worst}: baseline RMSE = {rmse_base[i_worst]:.4f}, '
      f'width-4x RMSE = {rmse_w4x[i_worst]:.4f}  '
      f'(Δ RMSE = {rmse_base[i_worst] - rmse_w4x[i_worst]:+.4f})')

truth_i = mp['oof_true'][i_worst]
mask_i  = mp['mask'][i_worst]
truth_m = np.where(mask_i, truth_i, np.nan)
res_b = np.where(mask_i, mp['oof_pred'][i_worst]  - truth_i, np.nan)
res_w = np.where(mask_i, w4x['oof_pred'][i_worst] - truth_i, np.nan)
vmax = float(np.nanmax(np.abs(np.stack([res_b, res_w]))))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
extent = [0, truth_i.shape[1], 0, truth_i.shape[0]]

ax = axes[0]
im = ax.imshow(truth_m, origin='lower', extent=extent, cmap='viridis',
               aspect='auto', vmin=0, vmax=1)
ax.set_title(f'OOF target  (design {i_worst})')
plt.colorbar(im, ax=ax, fraction=0.04)
ax.set_xlabel('N grid'); ax.set_ylabel('T grid')

ax = axes[1]
im = ax.imshow(res_b, origin='lower', extent=extent, cmap='RdBu_r',
               aspect='auto', vmin=-vmax, vmax=+vmax)
ax.set_title(f'baseline residual (RMSE = {rmse_base[i_worst]:.3f})')
plt.colorbar(im, ax=ax, fraction=0.04)
ax.set_xlabel('N grid'); ax.set_ylabel('T grid')

ax = axes[2]
im = ax.imshow(res_w, origin='lower', extent=extent, cmap='RdBu_r',
               aspect='auto', vmin=-vmax, vmax=+vmax)
ax.set_title(f'width-4x residual (RMSE = {rmse_w4x[i_worst]:.3f})')
plt.colorbar(im, ax=ax, fraction=0.04)
ax.set_xlabel('N grid'); ax.set_ylabel('T grid')

plt.tight_layout()
plt.show()

# Population summary too.
delta = rmse_base - rmse_w4x
print(f'\nAcross all 2511 designs (positive Δ = width-4x better):')
print(f'  mean   Δ RMSE = {np.nanmean(delta):+.4f}')
print(f'  median Δ RMSE = {np.nanmedian(delta):+.4f}')
print(f'  width-4x improved {(delta > 0).sum()} / {len(delta)} designs '
      f'({(delta > 0).mean()*100:.1f}%)')

## 7.8  Limits and next steps

- **Architecture work paid off.** The $R^2 \approx 0.91$ plateau identified at the
  baseline architecture was architectural, not data-limited (per §7.7). The
  width-4× variant (633 k parameters, OOF $R^2 = 0.973$) is the production decoder
  going forward; depth+1 is the parameter-efficient alternative at $R^2 = 0.963$
  using only baseline's 157 k parameters.
- **Future architecture work** *(deferred)*: §4.7.2 levers 3–4 — a 2-D
  self-attention head over the latent grid (long-range spatial coupling) and a
  U-Net-style residual decoder with a physics-derived prior (e.g. the Phase-2
  envelope mask). Marginal headroom is bounded above by ~0.03 in $R^2$ given the
  current plateau, so these are scoped as a follow-up iteration rather than a
  thesis-blocking item.
- **Re-evaluate Chapters 5 and 6 on the new pretrained weights.** The transfer-learning
  sweep in Chapter 5 currently uses the baseline `MapDecoder` fold-1 weights as
  the pretrained source; rerunning it with width-4× weights may shift the
  cross-domain story (and the η→PF transfer headline). The CV-fold ensemble UQ in
  Chapter 6 will likewise tighten with the better-fit decoder.
- **$N_\text{max}$ is constant** in the current dataset (all designs share the same
  DC-bus voltage and current limit). Adding designs with varying voltage limits would
  let the envelope net predict both axes meaningfully.
- **Iron loss model is lumped**: one peak air-gap $B$ per operating point. A proper
  Bertotti/Steinmetz needs rotor-stepped per-element integration; both nets would
  re-train trivially on the improved dataset since the I/O shapes don't change.
- **FSCW topology — cross-topology transfer landed (2026-05-27).** The
  `mi_createmesh()` crash that blocked the earlier FSCW LHS sweep was traced to
  magnet outer-corner overhang past the rotor OD and fixed on the geometry side.
  The FSCW master dataset now holds 49 valid designs, and the IPM→FSCW transfer
  story is written up in §5.7–5.9 of Chapter 5: the IPM η `MapDecoder`
  `pretrained-naive` mode wins at every $n_\text{train} \ge 10$, peaking at
  $R^2 = 0.84$ at $n=35$. Standalone FSCW 5-fold CV (no transfer) on the same
  49 designs hits $R^2 = 0.835$ — *both methods plateau at the same accuracy*,
  consistent with the §4.7.3 capacity-bound finding on the IPM side.
- **Uncertainty quantification**: the OOF predictions land within ±1 Nm for
  $T_\text{max}$ but there's no per-prediction confidence interval yet from the
  point-estimate networks alone. The CV-fold ensemble in Chapter 6 supplies one
  honest UQ signal already; the §6.8 upgrades (independent ensemble, MC-Dropout,
  quantile heads) would tighten it further.
